# Lab 1 — Engagement Prediction

**Author:** Kravchenko Timur, group M3336
**Course:** Social Media Analytics, Agents & LLMs — ITMO, autumn 2026
**Metric:** macro-F1 (primary), QWK, accuracy

## Task

Three-class ordinal classification of LinkedIn posts relative to the author's own typical engagement:
- `0 = below` — post performed worse than the author's usual
- `1 = typical` — on par with the usual
- `2 = above` — better than the author's usual

Label definition (frozen in the dataset):
- `level = median(log(1 + reactions))` over the author's **earlier** posts (min 5)
- `resid = log(1 + reactions) − level`
- Cutoffs `low=-0.549306, high=+1.157453` — 20th/80th percentiles of train residuals.

Split: 80/20 **by author** (seed 42) — 3295 train / 824 test authors, 71395 / 17623 posts.

## Pipeline overview

1. **Tier 1** — author-relative tabular features on top of the baseline.
2. **Tier 2** — text embeddings from `multilingual-e5-small` + PCA + style stats.
3. **Tier 3** — CLIP image embeddings (Colab T4 GPU, OpenCLIP ViT-B/32).
4. **Tier 4** — ordinal regression on `resid`, LR stacker, Optuna tuning.
5. **Blend** — weighted average of Tier 2 and Tier 3 probabilities.

All modules live in `my/`. Data is read via the `data/` symlink to the unpacked Kairos release.

**Submission:** Blend (40% Tier 2 + 60% Tier 3), `results/predictions/supertimur.csv`.

## 1. Loading and first look

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

from kairos.data import load_labels, load_split, load_followers
from kairos.label import author_level

labels = load_labels()
split = load_split()
followers = load_followers()
print(f'labels: {len(labels):,} rows')
print(f'train authors: {len(split["train"]):,}, test authors: {len(split["test"]):,}')
print(f'cutoffs: low={split["cutoffs_on_log_residual"]["low"]:+.6f}, high={split["cutoffs_on_log_residual"]["high"]:+.6f}')

### 1.1 Class distribution

By construction the balance is roughly 20 / 60 / 20 — classes are defined by the 20th and 80th percentiles of residuals. `macro-F1` is sensitive to the minority classes (`below` and `above`), so a model with `class_weight="balanced"` is justified.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for i, split_name in enumerate(['train', 'test']):
    sub = labels[labels['split'] == split_name]
    counts = sub['class'].value_counts().reindex(['below', 'typical', 'above'])
    ax[i].bar(counts.index, counts.values, color=['#d9484f', '#8c9eff', '#58b65c'])
    for j, v in enumerate(counts.values):
        ax[i].text(j, v + 50, f'{v:,}\n({v/len(sub)*100:.1f}%)', ha='center')
    ax[i].set_title(f'{split_name}  (n={len(sub):,})')
    ax[i].set_ylabel('posts')
plt.tight_layout()
plt.show()

### 1.2 `log_reactions` distribution and author profile

In [ ]:
# Load precomputed Tier 1 cache instead of re-parsing 5433 JSON feeds
tier1 = pd.read_parquet('cache/tier1_features.parquet')
lbl_t = tier1.merge(labels, on=['author', 'post'], how='inner')

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for cls, color in [('below', '#d9484f'), ('typical', '#8c9eff'), ('above', '#58b65c')]:
    sub = lbl_t[lbl_t['class'] == cls]
    ax[0].hist(sub['log_reactions'], bins=60, alpha=0.5, label=cls, color=color)
ax[0].set_xlabel('log(1 + reactions)')
ax[0].set_ylabel('posts')
ax[0].set_title('log_reactions distribution per class')
ax[0].legend()

follower_log = np.log10(followers.clip(lower=1))
ax[1].hist(follower_log.dropna(), bins=50, color='#2d7a3d', alpha=0.8)
ax[1].set_xlabel('log10(followers)')
ax[1].set_ylabel('authors')
ax[1].set_title(f'Followers across {len(followers):,} authors')
plt.tight_layout()
plt.show()

### 1.3 Author activity and content types

In [ ]:
posts_per_author = tier1.groupby('author').size()
ctype = tier1['content_type'].value_counts().head(8)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(posts_per_author, bins=np.logspace(0, 3, 40), color='#4a6fa5')
ax[0].set_xscale('log')
ax[0].set_xlabel('posts per author (log)')
ax[0].set_ylabel('authors')
ax[0].set_title(f'Activity: median {posts_per_author.median():.0f}, max {posts_per_author.max()}')

ctype.plot.barh(ax=ax[1], color='#4a6fa5')
ax[1].set_xlabel('posts')
ax[1].set_title('Content types (top 8)')
ax[1].invert_yaxis()
plt.tight_layout()
plt.show()

### 1.4 Class vs features — quick look

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
feats = [('n_chars', 'Post length (chars)', True),
         ('n_hashtags', 'Hashtags', False),
         ('log_followers', 'log(1 + followers)', False),
         ('days_since_prev', 'Days since previous post', False)]
for ax, (col, title, logx) in zip(axes.flat, feats):
    data = [lbl_t[lbl_t['class'] == c][col] for c in ['below', 'typical', 'above']]
    ax.boxplot(data, tick_labels=['below', 'typical', 'above'], showfliers=False)
    if logx:
        ax.set_yscale('log')
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 2. Features added on top of the baseline

### Tier 1: author-relative tabular (`my/features_tier1.py`)

Everything referencing author history uses `expanding + shift(1)` so a post never sees itself.

- **Author-relative deltas**: `n_chars_delta`, `n_words_delta`, `n_hashtags_delta`, `n_mentions_delta`, `n_urls_delta`, `is_textless_vs_author`, `has_question_vs_author`.
- **Cyclic time**: `hour_sin/cos`, `weekday_sin/cos`, `day_of_month_sin/cos`, `hour_delta_circular` against the author's typical hour direction.
- **`weekday_match`**: whether the weekday matches the author's modal weekday.
- **`content_type_rarity`**: share of this content type in the author's prior history.
- **Rolling reactions**: `roll_mean_5`, `roll_std_5`, `roll_mean_20`, `roll_mean_all`, `roll_std_all`, `roll_trend_5` (OLS slope over last 5).
- **Activity**: `posts_per_week_90d`, `cadence_cv_30d`, `days_since_first_post`, `hist_n_log`.
- **Artefacts**: `is_recent_post`, `age_days_log`.

### Tier 2: text (`my/features_tier2.py`)

- **Embeddings** from `intfloat/multilingual-e5-small` (384-dim, mean-pooled, L2-normalised), `max_seq_length=160`.
- **PCA-128** of embeddings (fit only on train).
- **Author centroid**: `cos_to_author`, `l2_to_author`, `author_posts_so_far` — how far the post is from the author's usual style.
- **Style**: `avg_word_len`, `caps_ratio`, `exclaim_count`, `question_count`, `emoji_count`, `bullet_count`, `placeholder_density`.

### Tier 3: images (`my/features_tier3.py`)

- **CLIP ViT-B/32** (OpenCLIP, `laion2b_s34b_b79k`) applied to 16 349 labelled posts that have a post image (~14.5%). GPU inference in a Colab T4 notebook (`notebooks/colab_clip_embeddings.ipynb`).
- **PCA-48** of 512-dim CLIP embeddings on train authors.
- **Author-relative image**: `has_image`, `img_cos_to_author`, `img_l2_to_author`, `img_author_count`.

## 3. Ablation — all runs

All models: LightGBM multiclass, class_weight="balanced", 5-fold GroupKFold by author.
Test metrics via `python evaluate.py --predictions results/predictions/<file>.csv`.

In [ ]:
RESULTS = Path('results')

rows = []
base = json.loads((RESULTS / 'baseline.json').read_text()) if (RESULTS / 'baseline.json').exists() else None
if base:
    for name, m in base['results'].items():
        rows.append({'tier': 'baseline', 'model': name, 'oof_f1': None,
                     'test_f1': m['macro_f1'], 'qwk': m['qwk'], 'acc': m['accuracy']})

for tier_file, prefix in [('tier1_cv.json', 'tier1'), ('tier2_cv.json', 'tier2'),
                          ('tier3_cv.json', 'tier3'), ('tier4_cv.json', 'tier4'),
                          ('blend_cv.json', 'blend')]:
    p = RESULTS / tier_file
    if not p.exists():
        continue
    data = json.loads(p.read_text())
    if tier_file == 'blend_cv.json':
        tm = data['test']
        rows.append({'tier': 'blend', 'model': f'blend_t2={data["best_weight_tier2"]:.1f}',
                     'oof_f1': None, 'test_f1': tm['macro_f1'],
                     'qwk': tm['qwk'], 'acc': tm['accuracy']})
    elif 'metrics' in data:
        for mn, info in data['metrics'].items():
            rows.append({'tier': prefix, 'model': mn,
                         'oof_f1': info['oof']['macro_f1'],
                         'test_f1': info['test']['macro_f1'],
                         'qwk': info['test']['qwk'],
                         'acc': info['test']['accuracy']})
    elif 'oof' in data:
        for mn, m in data['oof'].items():
            tm = data['test'][mn]
            rows.append({'tier': prefix, 'model': mn,
                         'oof_f1': m['macro_f1'],
                         'test_f1': tm['macro_f1'], 'qwk': tm['qwk'],
                         'acc': tm['accuracy']})

df_results = pd.DataFrame(rows)
df_results

## 4. Final model and predictions

In [ ]:
best_row = df_results.sort_values('test_f1', ascending=False).iloc[0]
print(f'Best: {best_row["tier"]}/{best_row["model"]}')
print(f'test macro-F1 = {best_row["test_f1"]:.4f}, QWK = {best_row["qwk"]:.4f}, acc = {best_row["acc"]:.4f}')

In [ ]:
# Confusion matrix of the submitted model
from sklearn.metrics import confusion_matrix, f1_score

best_csv = 'results/predictions/supertimur.csv'
pred = pd.read_csv(best_csv)
truth = labels[labels['split'] == 'test'][['author', 'post', 'label']]
merged = truth.merge(pred, on=['author', 'post'], suffixes=('_true', '_pred'))
cm = confusion_matrix(merged['label_true'], merged['label_pred'])

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=['below', 'typical', 'above'],
            yticklabels=['below', 'typical', 'above'], ax=ax)
ax.set_xlabel('predicted')
ax.set_ylabel('truth')
ax.set_title(f'Confusion matrix ({Path(best_csv).stem})')
plt.tight_layout()
plt.show()

f1_per_class = f1_score(merged['label_true'], merged['label_pred'], average=None)
print(f'F1 per class: below {f1_per_class[0]:.4f}, '
      f'typical {f1_per_class[1]:.4f}, above {f1_per_class[2]:.4f}')

## 5. Conclusions and ideas

### What worked
- **Author-relative tabular features** give the main lift (baseline 0.541 → tier1 0.610, +6.9%). Rolling statistics over log_reactions and deltas from the author's median are the main drivers — they directly reflect that the label is author-relative.
- **Text embeddings** via PCA-128 add +2.3% (tier1 0.610 → tier2 0.636). PCA-32 and PCA-128 are nearly indistinguishable — information past the first 32 components is marginal.
- **GroupKFold by author** is required; a vanilla KFold would leak history across folds.
- **Weighted blend of Tier 2 and Tier 3 probabilities** adds a small but consistent +0.24% on test (0.636 → 0.639), with best weight ~0.4/0.6 Tier 2 vs Tier 3.

### What did NOT work
- **Raw 384-dim embeddings** without PCA — worse than both PCA-32 and PCA-128. With 384 raw features, LightGBM dilutes the signal through column subsampling.
- **CatBoost** on this feature space is consistently 2-3% below LightGBM.
- **Ordinal regression** on `resid` has the highest accuracy but the lowest macro-F1 — too aggressive in predicting `typical`.
- **Stacking** via Logistic Regression with class_weight="balanced" on OOF probabilities hurt performance (double balancing on top of balanced base models + weaker base models dragging the stacker down).
- **CLIP images** alone did not improve over Tier 2 (0.636 → 0.636, essentially tied). Coverage of only 14.5% of labelled posts means the extra noise in the 85.5% without images cancels out the signal from the ones that do.

### Ideas for improvement
- **Video thumbnails** from the `lab1_video.*.7z` archive — not downloaded here due to disk constraints, likely +0.5-1 p.p.
- **Social graph features**: cross-mentions between authors, industry trends — likely +2-3 p.p. per the teacher's rough breakdown.
- **Weighted-average probability blending** with weights tuned on OOF macro-F1 instead of grid search.
- **Base-model calibration** (Platt scaling, isotonic) before stacking.